# kang study: CLAP audio + CLAP text + class MLP

이 노트북은 confidence 1-5 분류를 최대한 단순한 조건에서 확인하기 위한 실험입니다.

사용 feature:

- CLAP audio embedding, 512 dim
- CLAP text embedding, 512 dim
- second-level class one-hot

사용하지 않는 feature:

- title length, tag count, description length, has description
- class_top one-hot
- handcrafted cosine/L2/norm feature
- ensemble, stacking, two-tower

확인할 것:

- 5-fold OOF classification 성능
- Accuracy, balanced accuracy, macro F1, QWK, MAE, Spearman
- confusion matrix로 class가 얼마나 맞는지
- true confidence별 predicted score 분포
- MLP hidden embedding에서 confidence가 군집화되는지

In [ ]:
from pathlib import Path
import json
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    mean_absolute_error,
    precision_recall_fscore_support,
    silhouette_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
METADATA_DIR = DATA_DIR / "metadata"
FEATURE_DIR = DATA_DIR / "features"

OUT_DIR = ROOT / "outputs" / "kang_study"
REPORT_DIR = OUT_DIR / "reports"
PRED_DIR = OUT_DIR / "predictions"
PLOT_DIR = OUT_DIR / "plots"
CKPT_DIR = OUT_DIR / "checkpoints"
for path in [REPORT_DIR, PRED_DIR, PLOT_DIR, CKPT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": 42,
    "folds": 5,
    "epochs": 50,
    "patience": 7,
    "batch_size": 256,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "dropout": 0.3,
    "hidden": (512, 256),
    "loss_name": "emd",  # "emd" or "ce"
    "tsne_sample_n": 3000,
    "bsd10k_metadata": METADATA_DIR / "BSD10k_metadata.csv",
    "bsd35k_metadata": METADATA_DIR / "BSD35k-CS_metadata.csv",
    "bsd10k_audio_dir": FEATURE_DIR / "clap_audio_embeddings",
    "bsd10k_text_dir": FEATURE_DIR / "clap_text_embeddings",
    "bsd35k_audio_dir": FEATURE_DIR / "BSD35k_clap_audio_embeddings",
    "bsd35k_text_dir": FEATURE_DIR / "BSD35k-CS_clap_text_embeddings",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_VALUES_NP = np.arange(1, 6, dtype=np.float32)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("device:", DEVICE)
print("output dir:", OUT_DIR)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def clean_metadata(df, require_confidence):
    df = df.copy()
    df["sound_id"] = df["sound_id"].astype(str).str.strip()
    df["class"] = df["class"].astype(str).str.strip()
    if "class_top" not in df.columns:
        df["class_top"] = df["class"].str.split("-").str[0]
    else:
        df["class_top"] = df["class_top"].fillna(df["class"].str.split("-").str[0]).astype(str).str.strip()

    class_idx = df["class_idx"].astype(str).str.strip()
    keep = ~((class_idx.str.len() == 3) & (class_idx.str.endswith("99") | class_idx.str.endswith("00")))
    df = df[keep].copy()

    if require_confidence:
        df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce")
        df = df[df["confidence"].isin([1, 2, 3, 4, 5])].copy()
        df["confidence"] = df["confidence"].astype(int)

    return df.reset_index(drop=True)


def one_hot(values, categories):
    index = {cat: idx for idx, cat in enumerate(categories)}
    arr = np.zeros((len(values), len(categories)), dtype=np.float32)
    for row_idx, value in enumerate(values):
        col_idx = index.get(str(value))
        if col_idx is not None:
            arr[row_idx, col_idx] = 1.0
    return arr


def load_embeddings_and_df(df, audio_dir, text_dir):
    audio_rows = []
    text_rows = []
    kept = []

    for idx, row in df.reset_index(drop=True).iterrows():
        sound_id = str(row["sound_id"])
        audio_path = audio_dir / f"{sound_id}.npy"
        text_path = text_dir / f"{sound_id}.npy"
        if audio_path.is_file() and text_path.is_file():
            audio_rows.append(np.load(audio_path).reshape(-1).astype(np.float32))
            text_rows.append(np.load(text_path).reshape(-1).astype(np.float32))
            kept.append(idx)

    if not kept:
        raise RuntimeError("No rows had both audio and text embeddings.")

    kept_df = df.reset_index(drop=True).iloc[kept].reset_index(drop=True).copy()
    audio = np.vstack(audio_rows).astype(np.float32)
    text = np.vstack(text_rows).astype(np.float32)
    return kept_df, audio, text


def build_feature_matrix(df, audio_dir, text_dir, class_categories):
    kept_df, audio, text = load_embeddings_and_df(df, audio_dir, text_dir)
    class_x = one_hot(kept_df["class"].astype(str).tolist(), class_categories)
    x = np.hstack([audio, text, class_x]).astype(np.float32)
    return kept_df, x


seed_everything(CONFIG["seed"])

In [ ]:
bsd10k_raw = pd.read_csv(CONFIG["bsd10k_metadata"])
bsd10k = clean_metadata(bsd10k_raw, require_confidence=True)
class_categories = sorted(bsd10k["class"].astype(str).unique().tolist())

bsd10k_df, x_raw = build_feature_matrix(
    bsd10k,
    CONFIG["bsd10k_audio_dir"],
    CONFIG["bsd10k_text_dir"],
    class_categories,
)
y_1based = bsd10k_df["confidence"].to_numpy(dtype=np.int64)
y_0based = y_1based - 1

print("rows:", len(bsd10k_df))
print("input_dim:", x_raw.shape[1])
print("feature dims: audio=512, text=512, class=", len(class_categories))
print("class categories:", class_categories)

dist = pd.Series(y_1based).value_counts().sort_index().rename_axis("confidence").reset_index(name="n")
dist["rate"] = dist["n"] / dist["n"].sum()
display(dist)

In [ ]:
class ConfidenceDataset(Dataset):
    def __init__(self, x, y=None):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        item = {"x": self.x[idx]}
        if self.y is not None:
            item["y"] = self.y[idx]
        return item


class ConfidenceMLP(nn.Module):
    def __init__(self, input_dim, n_classes=5, hidden=(512, 256), dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(input_dim, hidden[0]),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[0], hidden[1]),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden[1], n_classes)

    def forward_features(self, x):
        return self.features(x)

    def forward(self, x):
        return self.classifier(self.forward_features(x))


def expected_score_from_probs(probs):
    return (probs * LABEL_VALUES_NP.reshape(1, -1)).sum(axis=1)


def compute_loss(logits, y, loss_name="emd"):
    if loss_name == "ce":
        return F.cross_entropy(logits, y)

    if loss_name == "emd":
        probs = F.softmax(logits, dim=1)
        true = F.one_hot(y, num_classes=5).float()
        pred_cdf = probs.cumsum(dim=1)
        true_cdf = true.cumsum(dim=1)
        return torch.mean((pred_cdf - true_cdf) ** 2)

    raise ValueError(loss_name)


def predict_probs_and_features(model, loader):
    model.eval()
    probs = []
    feats = []
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(DEVICE)
            h = model.forward_features(x)
            logits = model.classifier(h)
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
            feats.append(h.cpu().numpy())
    return np.vstack(probs).astype(np.float32), np.vstack(feats).astype(np.float32)


print(ConfidenceMLP(x_raw.shape[1], hidden=CONFIG["hidden"], dropout=CONFIG["dropout"]))

In [ ]:
def metrics_from_probs(y_true_1based, probs):
    score = expected_score_from_probs(probs)
    pred_class = probs.argmax(axis=1) + 1
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_1based,
        pred_class,
        average="macro",
        zero_division=0,
    )
    rho = spearmanr(y_true_1based, score).statistic
    return {
        "mae_expected_score": float(mean_absolute_error(y_true_1based, score)),
        "spearman_expected_score": float(0.0 if np.isnan(rho) else rho),
        "accuracy": float(accuracy_score(y_true_1based, pred_class)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true_1based, pred_class)),
        "quadratic_weighted_kappa": float(cohen_kappa_score(y_true_1based, pred_class, weights="quadratic")),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
    }


def per_class_metrics_df(y_true, probs):
    pred = probs.argmax(axis=1) + 1
    report = classification_report(y_true, pred, labels=[1, 2, 3, 4, 5], output_dict=True, zero_division=0)
    rows = []
    for cls in [1, 2, 3, 4, 5]:
        item = report[str(cls)]
        rows.append({
            "confidence": cls,
            "precision": item["precision"],
            "recall": item["recall"],
            "f1": item["f1-score"],
            "support": int(item["support"]),
        })
    return pd.DataFrame(rows)


def train_one_fold(fold, train_idx, valid_idx):
    seed_everything(CONFIG["seed"] + fold)

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_raw[train_idx]).astype(np.float32)
    x_valid = scaler.transform(x_raw[valid_idx]).astype(np.float32)

    train_loader = DataLoader(
        ConfidenceDataset(x_train, y_0based[train_idx]),
        batch_size=CONFIG["batch_size"],
        shuffle=True,
    )
    valid_loader = DataLoader(ConfidenceDataset(x_valid), batch_size=CONFIG["batch_size"], shuffle=False)

    model = ConfidenceMLP(x_raw.shape[1], hidden=CONFIG["hidden"], dropout=CONFIG["dropout"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

    best_mae = math.inf
    best_epoch = -1
    best_state = None
    stale = 0
    history = []

    for epoch in range(CONFIG["epochs"]):
        model.train()
        total_loss = 0.0
        seen = 0

        for batch in train_loader:
            xb = batch["x"].to(DEVICE)
            yb = batch["y"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = compute_loss(model(xb), yb, CONFIG["loss_name"])
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * xb.shape[0]
            seen += xb.shape[0]

        scheduler.step()
        probs, _ = predict_probs_and_features(model, valid_loader)
        val_mae = mean_absolute_error(y_1based[valid_idx], expected_score_from_probs(probs))
        val_acc = accuracy_score(y_1based[valid_idx], probs.argmax(axis=1) + 1)
        history.append({"epoch": epoch, "train_loss": total_loss / max(seen, 1), "val_mae": float(val_mae), "val_accuracy": float(val_acc)})

        if val_mae < best_mae - 1e-6:
            best_mae = float(val_mae)
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= CONFIG["patience"]:
                break

    model.load_state_dict(best_state)
    probs, hidden = predict_probs_and_features(model, valid_loader)

    torch.save(
        {
            "model_state": best_state,
            "input_dim": int(x_raw.shape[1]),
            "scaler_mean": scaler.mean_.astype(np.float32),
            "scaler_scale": scaler.scale_.astype(np.float32),
            "class_categories": class_categories,
            "config": CONFIG,
            "best_epoch": int(best_epoch),
        },
        CKPT_DIR / f"fold_{fold}.pt",
    )
    pd.DataFrame(history).to_csv(REPORT_DIR / f"fold_{fold}_history.csv", index=False)

    return probs, hidden, {"fold": fold, "best_epoch": int(best_epoch), "best_mae": float(best_mae)}

In [ ]:
start = time.time()
splitter = StratifiedKFold(n_splits=CONFIG["folds"], shuffle=True, random_state=CONFIG["seed"])
splits = list(splitter.split(np.zeros(len(y_1based)), y_1based))

oof_probs = np.zeros((len(y_1based), 5), dtype=np.float32)
oof_hidden = np.zeros((len(y_1based), CONFIG["hidden"][1]), dtype=np.float32)
fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(splits):
    probs, hidden, fold_info = train_one_fold(fold, train_idx, valid_idx)
    oof_probs[valid_idx] = probs
    oof_hidden[valid_idx] = hidden
    fold_rows.append(fold_info)
    print(f"fold={fold} mae={fold_info['best_mae']:.4f} best_epoch={fold_info['best_epoch']}")

cv_metrics = metrics_from_probs(y_1based, oof_probs)
cv_metrics.update({
    "rows": int(len(y_1based)),
    "input_dim": int(x_raw.shape[1]),
    "audio_dim": 512,
    "text_dim": 512,
    "class_dim": int(len(class_categories)),
    "loss_name": CONFIG["loss_name"],
    "elapsed_seconds": float(time.time() - start),
})

fold_df = pd.DataFrame(fold_rows)
summary_df = pd.DataFrame([cv_metrics])
per_class_df = per_class_metrics_df(y_1based, oof_probs)

display(summary_df)
display(fold_df)
display(per_class_df)

summary_df.to_csv(REPORT_DIR / "kang_study_cv_summary.csv", index=False)
fold_df.to_csv(REPORT_DIR / "kang_study_fold_metrics.csv", index=False)
per_class_df.to_csv(REPORT_DIR / "kang_study_per_class_metrics.csv", index=False)

with open(REPORT_DIR / "kang_study_summary.json", "w", encoding="utf-8") as f:
    json.dump(cv_metrics, f, indent=2, ensure_ascii=False)

print(json.dumps(cv_metrics, indent=2, ensure_ascii=False))

In [ ]:
oof_out = bsd10k_df.copy()
oof_out["true_confidence"] = y_1based
oof_out["predicted_confidence_class"] = oof_probs.argmax(axis=1) + 1
oof_out["predicted_confidence_score"] = expected_score_from_probs(oof_probs)
for idx in range(5):
    oof_out[f"prob_confidence_{idx + 1}"] = oof_probs[:, idx]

oof_path = PRED_DIR / "BSD10k_oof_kang_study.csv"
oof_out.to_csv(oof_path, index=False)
np.save(PRED_DIR / "BSD10k_oof_hidden_kang_study.npy", oof_hidden)

cm = pd.DataFrame(
    confusion_matrix(y_1based, oof_out["predicted_confidence_class"], labels=[1, 2, 3, 4, 5]),
    index=[f"true_{i}" for i in range(1, 6)],
    columns=[f"pred_{i}" for i in range(1, 6)],
)
cm.to_csv(REPORT_DIR / "kang_study_confusion_matrix.csv")

print("saved:", oof_path)
display(cm)

## Visualizations: classification quality

In [ ]:
def savefig(name):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    print("saved:", path)


# Confusion matrix: raw count
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(5), labels=[1, 2, 3, 4, 5])
ax.set_yticks(range(5), labels=[1, 2, 3, 4, 5])
ax.set_xlabel("Predicted confidence")
ax.set_ylabel("True confidence")
ax.set_title("OOF confusion matrix, count")
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(cm.values[i, j]), ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
savefig("confusion_matrix_count.png")
plt.show()


# Confusion matrix: row normalized recall view
cm_norm = cm.values / np.maximum(cm.values.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(5), labels=[1, 2, 3, 4, 5])
ax.set_yticks(range(5), labels=[1, 2, 3, 4, 5])
ax.set_xlabel("Predicted confidence")
ax.set_ylabel("True confidence")
ax.set_title("OOF confusion matrix, row normalized")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
savefig("confusion_matrix_row_normalized.png")
plt.show()

In [ ]:
# Per-class precision / recall / F1
fig, ax = plt.subplots(figsize=(8, 5))
xpos = np.arange(len(per_class_df))
width = 0.25
ax.bar(xpos - width, per_class_df["precision"], width=width, label="precision")
ax.bar(xpos, per_class_df["recall"], width=width, label="recall")
ax.bar(xpos + width, per_class_df["f1"], width=width, label="f1")
ax.set_xticks(xpos, labels=per_class_df["confidence"].astype(str))
ax.set_ylim(0, 1)
ax.set_xlabel("Confidence class")
ax.set_ylabel("Score")
ax.set_title("Per-class classification metrics")
ax.legend()
savefig("per_class_metrics.png")
plt.show()


# Predicted score distribution by true confidence
fig, ax = plt.subplots(figsize=(8, 5))
data = [oof_out.loc[oof_out["true_confidence"] == cls, "predicted_confidence_score"].values for cls in [1, 2, 3, 4, 5]]
ax.boxplot(data, labels=[1, 2, 3, 4, 5], showfliers=False)
ax.set_xlabel("True confidence")
ax.set_ylabel("Predicted expected confidence score")
ax.set_title("Predicted score distribution by true class")
savefig("predicted_score_by_true_confidence_boxplot.png")
plt.show()


# Prediction class distribution vs true class distribution
true_counts = pd.Series(y_1based).value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
pred_counts = pd.Series(oof_out["predicted_confidence_class"]).value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
fig, ax = plt.subplots(figsize=(8, 5))
xpos = np.arange(5)
ax.bar(xpos - 0.18, true_counts.values, width=0.36, label="true")
ax.bar(xpos + 0.18, pred_counts.values, width=0.36, label="predicted")
ax.set_xticks(xpos, labels=[1, 2, 3, 4, 5])
ax.set_xlabel("Confidence class")
ax.set_ylabel("Count")
ax.set_title("True vs predicted class distribution")
ax.legend()
savefig("true_vs_predicted_class_distribution.png")
plt.show()

## Visualizations: hidden embedding clustering

`oof_hidden`은 각 fold에서 validation sample을 통과시켜 얻은 MLP 마지막 hidden layer 256-d embedding입니다. 즉 학습에 직접 본 sample이 아니라 OOF 방식으로 얻은 hidden representation입니다.

In [ ]:
# PCA clustering view
pca = PCA(n_components=2, random_state=CONFIG["seed"])
hidden_pca = pca.fit_transform(oof_hidden)

pca_df = pd.DataFrame({
    "pc1": hidden_pca[:, 0],
    "pc2": hidden_pca[:, 1],
    "true_confidence": y_1based,
    "predicted_confidence_class": oof_out["predicted_confidence_class"].to_numpy(),
    "predicted_confidence_score": oof_out["predicted_confidence_score"].to_numpy(),
})
pca_df.to_csv(REPORT_DIR / "kang_study_hidden_pca.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
scatter = axes[0].scatter(pca_df["pc1"], pca_df["pc2"], c=pca_df["true_confidence"], s=8, alpha=0.55, cmap="viridis", vmin=1, vmax=5)
axes[0].set_title("Hidden PCA colored by true confidence")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
fig.colorbar(scatter, ax=axes[0], ticks=[1, 2, 3, 4, 5])

scatter = axes[1].scatter(pca_df["pc1"], pca_df["pc2"], c=pca_df["predicted_confidence_class"], s=8, alpha=0.55, cmap="viridis", vmin=1, vmax=5)
axes[1].set_title("Hidden PCA colored by predicted class")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
fig.colorbar(scatter, ax=axes[1], ticks=[1, 2, 3, 4, 5])
savefig("hidden_pca_true_vs_predicted.png")
plt.show()

print("PCA explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
# t-SNE clustering view. This can take a bit on CPU, so it samples rows.
rng = np.random.default_rng(CONFIG["seed"])
sample_n = min(CONFIG["tsne_sample_n"], len(oof_hidden))
sample_idx = rng.choice(len(oof_hidden), size=sample_n, replace=False)

tsne = TSNE(n_components=2, perplexity=35, learning_rate="auto", init="pca", random_state=CONFIG["seed"])
hidden_tsne = tsne.fit_transform(oof_hidden[sample_idx])

tsne_df = pd.DataFrame({
    "tsne1": hidden_tsne[:, 0],
    "tsne2": hidden_tsne[:, 1],
    "true_confidence": y_1based[sample_idx],
    "predicted_confidence_class": oof_out["predicted_confidence_class"].to_numpy()[sample_idx],
    "predicted_confidence_score": oof_out["predicted_confidence_score"].to_numpy()[sample_idx],
})
tsne_df.to_csv(REPORT_DIR / "kang_study_hidden_tsne_sample.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
scatter = axes[0].scatter(tsne_df["tsne1"], tsne_df["tsne2"], c=tsne_df["true_confidence"], s=10, alpha=0.7, cmap="viridis", vmin=1, vmax=5)
axes[0].set_title("Hidden t-SNE colored by true confidence")
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")
fig.colorbar(scatter, ax=axes[0], ticks=[1, 2, 3, 4, 5])

scatter = axes[1].scatter(tsne_df["tsne1"], tsne_df["tsne2"], c=tsne_df["predicted_confidence_class"], s=10, alpha=0.7, cmap="viridis", vmin=1, vmax=5)
axes[1].set_title("Hidden t-SNE colored by predicted class")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
fig.colorbar(scatter, ax=axes[1], ticks=[1, 2, 3, 4, 5])
savefig("hidden_tsne_true_vs_predicted.png")
plt.show()

In [ ]:
# Numeric clustering diagnostics. Higher silhouette means cleaner separation.
# Because confidence 1 is tiny, treat this as a rough diagnostic, not a final metric.
cluster_rows = []
try:
    cluster_rows.append({
        "space": "hidden_256",
        "label": "true_confidence",
        "silhouette": float(silhouette_score(oof_hidden, y_1based, metric="euclidean")),
    })
except Exception as exc:
    cluster_rows.append({"space": "hidden_256", "label": "true_confidence", "silhouette": np.nan, "error": str(exc)})

try:
    cluster_rows.append({
        "space": "hidden_256",
        "label": "predicted_confidence_class",
        "silhouette": float(silhouette_score(oof_hidden, oof_out["predicted_confidence_class"], metric="euclidean")),
    })
except Exception as exc:
    cluster_rows.append({"space": "hidden_256", "label": "predicted_confidence_class", "silhouette": np.nan, "error": str(exc)})

cluster_df = pd.DataFrame(cluster_rows)
cluster_df.to_csv(REPORT_DIR / "kang_study_clustering_metrics.csv", index=False)
display(cluster_df)

## Optional: train on all BSD10k and predict BSD35k-CS

In [ ]:
RUN_BSD35K_PREDICTION = True


def train_final_full_model(train_epochs):
    seed_everything(CONFIG["seed"])
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_raw).astype(np.float32)
    loader = DataLoader(ConfidenceDataset(x_train, y_0based), batch_size=CONFIG["batch_size"], shuffle=True)

    model = ConfidenceMLP(x_raw.shape[1], hidden=CONFIG["hidden"], dropout=CONFIG["dropout"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(train_epochs, 1))

    for epoch in range(train_epochs):
        model.train()
        for batch in loader:
            xb = batch["x"].to(DEVICE)
            yb = batch["y"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = compute_loss(model(xb), yb, CONFIG["loss_name"])
            loss.backward()
            optimizer.step()
        scheduler.step()

    torch.save(
        {
            "model_state": model.state_dict(),
            "input_dim": int(x_raw.shape[1]),
            "scaler_mean": scaler.mean_.astype(np.float32),
            "scaler_scale": scaler.scale_.astype(np.float32),
            "class_categories": class_categories,
            "config": CONFIG,
            "train_epochs": int(train_epochs),
        },
        CKPT_DIR / "final_full_bsd10k.pt",
    )
    return model, scaler


if RUN_BSD35K_PREDICTION:
    best_epochs = [row["best_epoch"] + 1 for row in fold_rows]
    final_epochs = int(round(float(np.median(best_epochs))))
    final_epochs = max(1, min(CONFIG["epochs"], final_epochs))
    print("final train epochs:", final_epochs)

    final_model, final_scaler = train_final_full_model(final_epochs)

    bsd35k_raw = pd.read_csv(CONFIG["bsd35k_metadata"])
    bsd35k = clean_metadata(bsd35k_raw, require_confidence=False)
    bsd35k_df, bsd35k_x_raw = build_feature_matrix(bsd35k, CONFIG["bsd35k_audio_dir"], CONFIG["bsd35k_text_dir"], class_categories)
    bsd35k_x = final_scaler.transform(bsd35k_x_raw).astype(np.float32)
    bsd35k_loader = DataLoader(ConfidenceDataset(bsd35k_x), batch_size=CONFIG["batch_size"], shuffle=False)
    bsd35k_probs, bsd35k_hidden = predict_probs_and_features(final_model, bsd35k_loader)

    bsd35k_out = bsd35k_df.copy()
    bsd35k_out["predicted_confidence_class"] = bsd35k_probs.argmax(axis=1) + 1
    bsd35k_out["predicted_confidence_score"] = expected_score_from_probs(bsd35k_probs)
    for idx in range(5):
        bsd35k_out[f"prob_confidence_{idx + 1}"] = bsd35k_probs[:, idx]

    bsd35k_path = PRED_DIR / "BSD35k-CS_predicted_kang_study.csv"
    bsd35k_out.to_csv(bsd35k_path, index=False)
    np.save(PRED_DIR / "BSD35k-CS_hidden_kang_study.npy", bsd35k_hidden)

    print("saved:", bsd35k_path)
    print("rows:", len(bsd35k_out))
    print("mean score:", float(bsd35k_out["predicted_confidence_score"].mean()))
    display(bsd35k_out["predicted_confidence_class"].value_counts().sort_index().rename_axis("predicted_class").reset_index(name="n"))